# ROCKET Family Models for Shaft Force Sensing

This notebook demonstrates how to train and evaluate ROCKET regression models (RocketRegressor, MultiRocketRegressor, and HydraRegressor) from the aeon-toolkit for shaft force sensing.

ROCKET models use random convolutional kernels to extract features from time series data, offering fast training times and competitive accuracy.

## 1. Load Libraries

In [1]:
import numpy as np
import plotly.graph_objs as go
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from itertools import chain
from datetime import datetime
from torch.utils.data import ConcatDataset, DataLoader, random_split

from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_error,
)
import time


In [2]:

from shaft_force_sensing import ForceSensingDataset
from shaft_force_sensing.models import (
    LitRocket,
    LitMultiRocket,
    LitHydra,
)
from shaft_force_sensing.evaluation import (
    tb_to_numpy,
    add_norm,
    array_bais,
    array_medfilt,
)

%load_ext autoreload
%autoreload 2

## 2. Set Configuration

In [3]:
# Configuration
batch_size = 256
random_seed = 42

# Data configuration

# Input and target column definitions
i_cols = [
    'jaw_position', 'wrist_pitch_position', 'wrist_yaw_position', 'roll_position',
    'wrist_pitch_velocity', 'wrist_yaw_velocity', 'jaw_velocity', 'roll_velocity',
    'wrist_pitch_effort', 'wrist_yaw_effort', 'roll_effort',
    'jaw_effort', 'insertion_effort', 'yaw_effort', 'pitch_effort',
    'tx', 'ty', 'tz', 'fx', 'fy', 'fz'
]
t_cols = ['ati_fx', 'ati_fy', 'ati_fz']

## 3. Load and Prepare Data

In [4]:
# Load datasets - update paths as needed
# data_dirs = [Path("path/to/data1"), Path("path/to/data2"), ...]

# For demonstration, create synthetic data or load your actual data
# datasets = [ForceSensingDataset(d, i_cols, t_cols) for d in data_dirs]
# dataset = ConcatDataset(datasets)

# Split into train and validation
# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create data loaders
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("TODO: Load your actual data following the ForceSensingDataset pattern")
print(f"Expected input columns: {len(i_cols)}")
print(f"Expected target columns: {len(t_cols)}")

TODO: Load your actual data following the ForceSensingDataset pattern
Expected input columns: 21
Expected target columns: 3


In [5]:
data_paths = sorted(Path("/home/sxk2514/erie/shaft_force_sensing/data/").rglob("*.csv"))

groups = defaultdict(list)
for p in data_paths:
    groups[p.parent.name].append(p)

test_paths = [lst[-1] for lst in groups.values()]
train_paths = [p for p in data_paths if p not in test_paths]
train_paths.pop(3);
train_paths.pop(2);

In [6]:
golbal_scaler = StandardScaler()
forces = []
for p in tqdm(train_paths):
    data = np.loadtxt(p, delimiter=",", skiprows=1)
    forces.append(data[:, -3:])
forces = np.concatenate(forces, axis=0)
golbal_scaler.fit(forces);

  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:21<00:00,  1.18it/s]


In [7]:
train_sets = defaultdict(list)
for p in tqdm(train_paths):
    stride = 5
    if p.parent.name == 'Free':
        stride *= 4
    dataset = ForceSensingDataset(
        p, i_cols, t_cols,
        stride, nomalizer=golbal_scaler)
    train_sets[p.parent.name].append(dataset)

train_set = ConcatDataset(
    list(chain.from_iterable(train_sets.values())))

100%|██████████| 25/25 [00:10<00:00,  2.29it/s]


In [8]:
train_size = int(0.9 * len(train_set))
val_size = len(train_set) - train_size
train_set, val_set = random_split(train_set, [train_size, val_size])

In [9]:
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

In [10]:
X, y, mask = next(iter(train_loader))
print(f"X shape: {X.shape}")   # [256, 100, 21] — 256 samples, 100 timesteps, 21 channels
print(f"y shape: {y.shape}")   # [256, 3]        — 3 force components
print(f"mask shape: {mask.shape}")  # [256, 100]

X shape: torch.Size([256, 100, 21])
y shape: torch.Size([256, 3])
mask shape: torch.Size([256, 100])


## 4. Train ROCKET Models

Train different ROCKET variants and compare their performance.

In [11]:
import inspect
from aeon.regression.convolution_based import RocketRegressor  # or sktime equivalent
print(inspect.signature(RocketRegressor.__init__))

(self, n_kernels=10000, estimator=None, random_state=None, n_jobs=1)


In [12]:
# Model configurations to compare
models_config = {
    'RocketRegressor': {
        'class': LitRocket,
        'kwargs': {
            'n_kernels': 2000,
            'n_jobs': 6   # ✅ add this
        },
        'description': 'ROCKET with 10K kernels'
    },
    'MultiRocketRegressor': {
        'class': LitMultiRocket,
        'kwargs': {
            'n_kernels': 2000,
            'n_jobs': 6   # ✅ add here too
        },
        'description': 'Multi-scale ROCKET with 5K kernels'
    },
    'HydraRegressor': {
        'class': LitHydra,
        'kwargs': {
            'n_kernels': 8,
            'n_groups': 64,
            'n_jobs': 6   # ⚠️ only if Hydra supports it
        },
        'description': 'Hydra with 8 kernels x 64 groups'
    },
}

results = {}


In [13]:
import time
import joblib
from pathlib import Path
from datetime import datetime

# Create experiment directory
save_root = Path("../logs")
save_dir = save_root / datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving models to: {save_dir}")
results = {}
best_model = None
best_loss = float("inf")

for model_name, config in models_config.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}: {config['description']}")
    print(f"{'='*60}")
    
    # Initialize model
    model = config['class'](
        d_input=len(i_cols),
        d_output=len(t_cols),
        **config['kwargs']
    )
    
    # -------------------------
    # ✅ TRAIN (returns val_loss)
    # -------------------------
    start_time = time.time()
    val_loss = model.train_rocket(train_loader, val_loader)  # 👈 IMPORTANT
    training_time = time.time() - start_time
    
    print(f"\nTraining completed in {training_time:.2f} seconds")
    
    if val_loss is not None:
        print(f"Validation Loss: {val_loss:.6f}")
    
    # -------------------------
    # ✅ SAVE MODEL
    # -------------------------
    model_path = save_dir / f"{model_name}.pkl"
    joblib.dump(model, model_path)
    print(f"Saved model to: {model_path}")
    
    # -------------------------
    # ✅ TRACK BEST MODEL
    # -------------------------
    if val_loss is not None and val_loss < best_loss:
        best_loss = val_loss
        best_model = model
        joblib.dump(model, save_dir / "best_model.pkl")
        print(f"🔥 New best model saved (val_loss={val_loss:.6f})")

    # Store results
    results[model_name] = {
        'model': model,
        'training_time': training_time,
        'val_loss': val_loss,
        'config': config,
        'model_path': model_path
    }

print("\n✅ All models trained and saved successfully!")

Saving models to: ../logs/20260325_115308

Training RocketRegressor: ROCKET with 10K kernels


Training ROCKET model with 370118 samples...


: 

: 

## 5. Evaluate Models on Test Set

In [1]:
import numpy as np
import joblib
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import (
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_error,
)
from collections import defaultdict

from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler 
from shaft_force_sensing import ForceSensingDataset
from tqdm import tqdm
from itertools import chain
from shaft_force_sensing.models import (
    LitRocket,
    LitMultiRocket,
    LitHydra,
)

from torch.utils.data import ConcatDataset, DataLoader, random_split
from shaft_force_sensing.evaluation import (
    tb_to_numpy,
    add_norm,
    array_bais,
    array_medfilt,
)
import joblib
import plotly.graph_objs as go


In [25]:
model_path  = Path("../logs/20260326_072309/RocketRegressor.pkl")
model = joblib.load(model_path)
save_path   = Path(".")


In [13]:
# Configuration
batch_size = 256
random_seed = 42

# Input and target column definitions
i_cols = [
    'jaw_position', 'wrist_pitch_position', 'wrist_yaw_position', 'roll_position',
    'wrist_pitch_velocity', 'wrist_yaw_velocity', 'jaw_velocity', 'roll_velocity',
    'wrist_pitch_effort', 'wrist_yaw_effort', 'roll_effort',
    'jaw_effort', 'insertion_effort', 'yaw_effort', 'pitch_effort',
    'tx', 'ty', 'tz', 'fx', 'fy', 'fz'
]
t_cols = ['ati_fx', 'ati_fy', 'ati_fz']

data_paths = sorted(Path("/home/sxk2514/erie/shaft_force_sensing/data/").rglob("*.csv"))

groups = defaultdict(list)
for p in data_paths:
    groups[p.parent.name].append(p)

test_paths = [lst[-1] for lst in groups.values()]
train_paths = [p for p in data_paths if p not in test_paths]
train_paths.pop(3);
train_paths.pop(2);

golbal_scaler = StandardScaler()
forces = []
for p in tqdm(train_paths):
    data = np.loadtxt(p, delimiter=",", skiprows=1)
    forces.append(data[:, -3:])
forces = np.concatenate(forces, axis=0)
golbal_scaler.fit(forces);

train_sets = defaultdict(list)
for p in tqdm(train_paths):
    stride = 5
    if p.parent.name == 'Free':
        stride *= 4
    dataset = ForceSensingDataset(
        p, i_cols, t_cols,
        stride, nomalizer=golbal_scaler)
    train_sets[p.parent.name].append(dataset)

train_set = ConcatDataset(
    list(chain.from_iterable(train_sets.values())))

train_size = int(0.9 * len(train_set))
val_size = len(train_set) - train_size
train_set, val_set = random_split(train_set, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)



  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:11<00:00,  2.14it/s]


In [26]:
test_metrics = {}
model_name   = model_path.stem
print(f"\nEvaluating {model_name}...")
 
all_preds = []
all_gts   = []
 
for batch in val_loader:
    x, y, _ = batch
    X_np = x.numpy() if hasattr(x, 'numpy') else np.array(x)
    y_np = y.numpy() if hasattr(y, 'numpy') else np.array(y)
 
    pred = model.rocket_model.predict(X_np)
    all_preds.append(pred)
    all_gts.append(y_np)
 
y_pred = np.vstack(all_preds)
y_gt   = np.vstack(all_gts).squeeze()
 
mse  = mean_squared_error(y_gt, y_pred)
rmse = root_mean_squared_error(y_gt, y_pred)
mae  = mean_absolute_error(y_gt, y_pred)
r2   = r2_score(y_gt, y_pred)
 
test_metrics[model_name] = {
    'mse':          mse,
    'rmse':         rmse,
    'mae':          mae,
    'r2':           r2,
    'predictions':  y_pred,
    'ground_truth': y_gt,
}
 
print(f"  MSE:  {mse:.6f}")
print(f"  RMSE: {rmse:.6f}")
print(f"  MAE:  {mae:.6f}")
print(f"  R2:   {r2:.6f}")



Evaluating RocketRegressor...


  MSE:  0.156174
  RMSE: 0.393854
  MAE:  0.275748
  R2:   0.860800


NameError: name 'axes' is not defined

In [27]:

axes        = ['Fx', 'Fy', 'Fz'] 
# ── Per-axis NRMSE + R2 metrics logged to file ────────────────────────────────
data = {model_name: (y_gt, y_pred)}
 
for group, (gt, pred) in data.items():
    gt_min   = np.min(gt, axis=0)
    gt_max   = np.max(gt, axis=0)
    gt_range = gt_max - gt_min
    rmse_per_axis  = root_mean_squared_error(gt, pred, multioutput='raw_values')
    nrmse          = rmse_per_axis / gt_range
    r2_scores      = r2_score(gt, pred, multioutput='raw_values')
 
    with open(save_path / "metrics.txt", "a") as f:
        print(f"Group: {group}", file=f)
        for i, name in enumerate(axes):
            print(
                f"{name}: "
                f"Range={gt_range[i]:.4f}, "
                f"RMSE={rmse_per_axis[i]:.4f}, "
                f"NRMSE={nrmse[i]*100:.2f}%, "
                f"R2={r2_scores[i]*100:.2f}",
                file=f)
        print("-" * 10, file=f)
 
print(f"\nMetrics saved to: {save_path / 'metrics.txt'}")
 



Metrics saved to: metrics.txt


In [28]:
# ── Plot Fx, Fy, Fz for all models ───────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=len(test_metrics),
    subplot_titles=[f"{name} — {label}"
                    for label in axes
                    for name in test_metrics.keys()],
    shared_xaxes=True,
)
 
for col, model_name in enumerate(test_metrics.keys(), 1):
    y_pred = test_metrics[model_name]['predictions']
    y_gt   = test_metrics[model_name]['ground_truth']
 
    for row, label in enumerate(axes, 1):
        fig.add_trace(
            go.Scatter(y=y_gt[:, row-1], name='Ground Truth',
                       mode='lines', legendgroup='gt',
                       showlegend=(row == 1 and col == 1)),
            row=row, col=col
        )
        fig.add_trace(
            go.Scatter(y=y_pred[:, row-1], name='Prediction',
                       mode='lines', legendgroup='pred',
                       showlegend=(row == 1 and col == 1)),
            row=row, col=col
        )
        fig.update_yaxes(title_text=f"{label} (N)", row=row, col=1)
 
fig.update_xaxes(title_text="Sample", row=3)
fig.update_layout(height=900, title_text="ROCKET Models: Force Prediction (Fx, Fy, Fz)")
fig.show()
 

